# Bookscan 검출기 파인튜닝 (Colab)

`Document-Scanner` 가중치를 **내가 찍은 책 사진**으로 이어 학습한다.

## 왜 필요한가

기본 가중치는 남이 '장면 속 문서'를 찾도록 학습시킨 것이다. 펼친 책의 **한 쪽**은
그 분포 밖이다 — 책등 너머에도 같은 종이가 이어지니, 모델 입장에서 거기 선을 그을
이유가 없다. 실제로 이 프로젝트가 부딪힌 실패가 정확히 그것이다.

책등 쪽 경계는 국소 기하로는 못 찾는다. 실측하면 경계 안쪽 15px에 잉크가 0.4%뿐이고
(빈 여백), 바깥은 그림자가 0.05에서 0.37로 매끄럽게 이어진다. 찾을 모서리도 따라갈
획도 없다. 규칙으로 짤 수 없고 **보여줘서 가르쳐야** 한다.

## 같이 바뀌는 것 두 가지

**해상도.** 원본은 256x256을 받는다. 4284x5712 사진이면 한 칸이 원본 22픽셀이라,
책등 선을 몇 픽셀 안에서 놓는 일에는 원리적으로 모자란다. 384로 올린다.
다만 해상도만 올리고 학습하지 않으면 (1,9)/(9,1) 스트립 합성곱의 수용영역이
상대적으로 좁아져 **오히려 나빠진다.** 그래서 둘은 반드시 세트다.

**확률 분포.** 원본 체크포인트는 확률을 0~1로 벌리지 못하고 0.47~0.72 안에 뭉쳐
놓는다. 그래서 bookscan은 고정 문턱 0.5를 못 쓰고 분포 안의 상대 위치로 자르는
우회를 넣어 두었다. BCE로 이어 학습하면 그 눌림이 펴진다.

---
## 순서

1. 런타임 유형을 **GPU**로 바꾼다 (메뉴 > 런타임 > 런타임 유형 변경 > T4 GPU)
2. 아래 셀을 위에서부터 차례로 실행
3. 자료(`kept.zip`)를 올린다
4. 학습이 끝나면 `.pth`를 내려받아 로컬 `models/` 에 넣는다


## 1. GPU 확인

GPU가 안 잡히면 런타임 유형을 바꾸고 **처음부터 다시** 실행해야 한다.


In [ ]:
import torch, time

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', DEVICE)
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))
else:
    print('[경고] GPU가 없습니다. 메뉴 > 런타임 > 런타임 유형 변경 > T4 GPU 로 바꾸고'
          ' 처음부터 다시 실행하세요. CPU로는 몇 시간씩 걸립니다.')


## 2. 자료 올리기

로컬에서 이렇게 만든 것을 올린다.

```bash
python tools/export_labels.py          # labels/ 로 뽑기
python -m labelcheck labels            # 훑어보고 나쁜 라벨 버리기
#   -> '유지한 것만 kept/로 모으기' 를 누르면 labels/kept/ 가 생긴다
```

그 `kept` 폴더를 **zip으로 압축해서** 아래 셀에 올린다.
(`labels` 폴더를 통째로 올려도 된다 — `review.json`이 있으면 버린 것은 알아서 뺀다.)


In [ ]:
import zipfile, json, shutil, random
from pathlib import Path
from google.colab import files

DATA = Path('/content/data')
if DATA.exists():
    shutil.rmtree(DATA)
DATA.mkdir(parents=True)

up = files.upload()          # kept.zip 을 고른다
for name in up:
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(name) as z:
            z.extractall(DATA)
    else:
        Path(name).replace(DATA / name)

# 압축을 풀면 폴더가 한 겹 더 생기는 일이 흔하다. 마스크가 실제로 있는 곳을 찾는다.
roots = {p.parent for p in DATA.rglob('*_mask.png')}
if not roots:
    raise SystemExit('*_mask.png 가 없습니다. export_labels.py 로 뽑은 폴더가 맞나요?')
ROOT = sorted(roots, key=lambda p: len(list(p.glob('*_mask.png'))))[-1]
print('자료 폴더:', ROOT)


## 3. 쌍 모으기

`review.json`에서 **버림**으로 표시한 것은 뺀다. 잘못된 라벨 한 장은 없는 것보다
나쁘다 — 모델이 그걸 정답이라고 배운다.


In [ ]:
def find_pairs(root):
    pairs = []
    for mask in sorted(root.glob('*_mask.png')):
        stem = mask.name[:-len('_mask.png')]
        for ext in ('.jpg', '.jpeg', '.png'):
            img = mask.with_name(stem + ext)
            if img.exists():
                pairs.append((img, mask, stem))
                break
    return pairs

pairs = find_pairs(ROOT)

review = ROOT / 'review.json'
if review.exists():
    verdicts = json.loads(review.read_text(encoding='utf-8'))
    before = len(pairs)
    pairs = [p for p in pairs if verdicts.get(p[2]) != 'drop']
    print('review.json: ' + str(before - len(pairs)) + '장을 버림으로 표시해 뺐습니다')

meta = {}
idx_file = ROOT / 'index.json'
if idx_file.exists():
    for row in json.loads(idx_file.read_text(encoding='utf-8')):
        meta[row.get('id')] = row

print('자료 ' + str(len(pairs)) + '쌍')
if meta:
    fixed = sum(1 for _, _, s in pairs if meta.get(s, {}).get('corrected'))
    print('  손으로 고친 것 ' + str(fixed) + '장, 자동 결과를 확인만 한 것 '
          + str(len(pairs) - fixed) + '장')
    if fixed > len(pairs) * 0.7:
        print('  [경고] 고친 사진에 치우쳐 있습니다. 모델이 책등 쪽을 넉넉히 잡는 것만'
              ' 배워서 원래 잘 되던 사진까지 배경을 물고 올 수 있습니다.')
if len(pairs) < 20:
    print('  [경고] 20쌍도 안 됩니다. 이 정도면 모델이 이 몇 장을 외울 뿐이라'
          ' 다른 사진에서 오히려 나빠집니다. 200~500쌍을 목표로 하세요.')


## 4. 모델 정의

**bookscan의 `bookscan/models/docscanner.py`와 글자 하나까지 같아야 한다.**
state_dict의 키가 모듈 속성 이름에서 나오기 때문에, 이름을 바꾸면 학습된
가중치를 못 불러온다.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

INPUT = 384          # 32의 배수여야 한다 (U-Net이 5번 풀링한다)
ENC_CHANNELS = [1, 64, 128, 256, 512, 512]
DEC_CHANNELS = [512, 512, 256, 128, 64, 1]
assert INPUT % 32 == 0

class ConvLayer(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=(3, 3), stride=1, padding='same'):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.layer(x)

class Encoder(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        # (1,9)와 (9,1) 두 방향의 긴 스트립 합성곱 - 문서의 긴 직선 모서리를 잡기 좋다
        self.spatial_conv_1 = ConvLayer(in_ch, out_ch, (1, 9))
        self.spatial_conv_2 = ConvLayer(in_ch, out_ch, (9, 1))
        self.out = ConvLayer(out_ch * 2, out_ch)

    def forward(self, x):
        return self.out(torch.concat((self.spatial_conv_1(x), self.spatial_conv_2(x)), dim=1))

class Decoder(nn.Module):
    def __init__(self, in_ch, out_ch, mid_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=3, stride=2,
                                     padding=1, bias=False)
        self.conv = nn.Sequential(ConvLayer(mid_ch, out_ch), ConvLayer(out_ch, out_ch))

    def forward(self, x, skip):
        x = self.up(x)
        # 전치 합성곱은 홀수 크기를 내놓아(8 -> 15) 스킵과 한 픽셀씩 어긋난다
        dy = skip.size()[2] - x.size()[2]
        dx = skip.size()[3] - x.size()[3]
        x = F.pad(x, [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2])
        return self.conv(torch.concat((x, skip), 1))

class Detector(nn.Module):
    def __init__(self, enc_channels, dec_channels):
        super().__init__()
        self.encoder_seq = nn.ModuleList(
            Encoder(enc_channels[i], enc_channels[i + 1])
            for i in range(len(enc_channels) - 1))
        self.decoder_seq = nn.ModuleList(
            Decoder(dec_channels[i], dec_channels[i + 1],
                    dec_channels[i] + dec_channels[i] // 2)
            for i in range(len(dec_channels) - 1))
        self.out = nn.Conv2d(1, 1, 1, padding='same')

    def forward(self, x):
        skips = []
        for layer in self.encoder_seq:
            x = layer(x)
            skips.append(x)
            x = F.avg_pool2d(x, 2)
        for layer, skip in zip(self.decoder_seq, skips[::-1]):
            x = layer(x, skip)
        return self.out(x)

print('모델 정의 완료')


## 5. 바탕 가중치 내려받기

남이 올린 `.pth`는 결국 pickle이라 임의 코드를 실행시킬 수 있다. SHA-256을
검증하고 `weights_only=True`로 텐서만 읽는다 (bookscan과 같은 방식).


In [ ]:
import hashlib, urllib.request

BASE_URL = 'https://huggingface.co/ordaktaktak/Document-Scanner/resolve/main/Scanner-Detector.pth'
BASE_SHA = 'f8c5809d06d49093581108bb71555c73e3f6044e3fcc5852682c1b78e9f8da10'
BASE = Path('/content/Scanner-Detector.pth')

if not BASE.exists():
    print('내려받는 중... (122MB)')
    urllib.request.urlretrieve(BASE_URL, BASE)

got = hashlib.sha256(BASE.read_bytes()).hexdigest()
if got != BASE_SHA:
    raise SystemExit('해시가 다릅니다: ' + got[:16] + '... 파일을 지우고 다시 받으세요.')
print('해시 확인됨')

model = Detector(ENC_CHANNELS, DEC_CHANNELS)
state = torch.load(BASE, map_location='cpu', weights_only=True)
if isinstance(state, dict) and 'state_dict' in state:
    state = state['state_dict']
model.load_state_dict(state)
model = model.to(DEVICE)
print('바탕 가중치 적재 완료')


## 6. 자료 읽기와 증강

**정규화를 bookscan의 `runtime.segment`와 똑같이 맞춘다.** 학습할 때와 추론할 때
입력 분포가 다르면 배운 것이 그대로 안 나온다 — 조용히 틀리는 종류라 나중에
원인 찾기가 고약하다.

증강은 가볍게 한다. 가르치려는 건 색이 아니라 **경계의 위치**고, 자료가 적을수록
원본에서 멀지 않은 변형이 안전하다.


In [ ]:
import cv2
import numpy as np

AUG_ROTATE, AUG_SCALE, AUG_BRIGHT, AUG_CONTRAST = 7.0, 0.12, 25.0, 0.20

def augment(img, mask, rng):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2),
                                rng.uniform(-AUG_ROTATE, AUG_ROTATE),
                                1.0 + rng.uniform(-AUG_SCALE, AUG_SCALE))
    img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_REPLICATE)
    mask = cv2.warpAffine(mask, M, (w, h), flags=cv2.INTER_NEAREST,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    a = 1.0 + rng.uniform(-AUG_CONTRAST, AUG_CONTRAST)
    b = rng.uniform(-AUG_BRIGHT, AUG_BRIGHT)
    return np.clip(img.astype(np.float32) * a + b, 0, 255).astype(np.uint8), mask

def prepare(img_path, mask_path, rng=None):
    gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if rng is not None:
        gray, mask = augment(gray, mask, rng)
    x = cv2.resize(gray, (INPUT, INPUT), interpolation=cv2.INTER_AREA).astype(np.float32)
    y = cv2.resize(mask, (INPUT, INPUT), interpolation=cv2.INTER_NEAREST)
    std = float(x.std())
    x = (x - x.mean()) / (std if std > 1e-6 else 1.0)   # runtime.segment 와 동일
    return x, (y > 127).astype(np.float32)

SEED = 0
rng = random.Random(SEED)
torch.manual_seed(SEED)

order = list(range(len(pairs)))
rng.shuffle(order)
n_val = max(1, int(len(pairs) * 0.2)) if len(pairs) > 1 else 0
val_set = [pairs[i] for i in order[:n_val]]
train_set = [pairs[i] for i in order[n_val:]]
print('학습 ' + str(len(train_set)) + ', 검증 ' + str(len(val_set))
      + '  |  입력 ' + str(INPUT) + 'px')

def batches(items, batch, rng):
    idxs = list(range(len(items)))
    if rng is not None:
        rng.shuffle(idxs)
    for i in range(0, len(idxs), batch):
        xs, ys = [], []
        for j in idxs[i:i + batch]:
            x, y = prepare(items[j][0], items[j][1], rng)
            xs.append(x)
            ys.append(y)
        yield (torch.from_numpy(np.stack(xs))[:, None].to(DEVICE),
               torch.from_numpy(np.stack(ys))[:, None].to(DEVICE))


## 7. 학습

검증 IoU가 가장 좋았을 때만 저장한다. 손실이 계속 줄어도 검증이 나빠지면
그건 외우고 있다는 뜻이다.


In [ ]:
EPOCHS = 40
BATCH = 8            # T4에서 384px 기준. 메모리가 부족하면 4로 줄인다.
LR = 1e-4
OUT = Path('/content/Scanner-Detector-ft.pth')

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
bce = torch.nn.BCEWithLogitsLoss()

def dice(logits, target):
    p = torch.sigmoid(logits)
    return 1.0 - (2 * (p * target).sum() + 1.0) / (p.sum() + target.sum() + 1.0)

def iou(logits, target):
    pred = (torch.sigmoid(logits) > 0.5).float()
    inter = (pred * target).sum().item()
    union = pred.sum().item() + target.sum().item() - inter
    return inter / union if union > 0 else 0.0

best = -1.0
for ep in range(1, EPOCHS + 1):
    model.train()
    t0, total, seen = time.time(), 0.0, 0
    for x, y in batches(train_set, BATCH, rng):
        opt.zero_grad(set_to_none=True)
        out = model(x)
        loss = bce(out, y) + dice(out, y)
        loss.backward()
        opt.step()
        total += float(loss.item()) * len(x)
        seen += len(x)
    sched.step()

    line = 'epoch ' + str(ep) + '/' + str(EPOCHS)
    line += '  loss ' + format(total / max(seen, 1), '.4f')
    if val_set:
        model.eval()
        with torch.no_grad():
            scores = [iou(model(x), y) for x, y in batches(val_set, BATCH, None)]
        m = float(np.mean(scores))
        line += '  val IoU ' + format(m, '.4f')
        if m > best:
            best = m
            torch.save(model.state_dict(), OUT)
            line += '  <- 저장'
    print(line + '  (' + format(time.time() - t0, '.1f') + 's)')

if best < 0:
    torch.save(model.state_dict(), OUT)
    print('검증 자료가 없어 마지막 상태를 저장했습니다.')
else:
    print('가장 좋았던 상태: val IoU ' + format(best, '.4f'))


## 8. 눈으로 확인

숫자만 보고 넘어가면 안 된다. IoU가 높아도 **책등 쪽 한 변만** 계속 틀릴 수 있고,
그게 정확히 우리가 고치려던 것이다. 초록이 정답, 빨강이 예측이다.


In [ ]:
import matplotlib.pyplot as plt

model.load_state_dict(torch.load(OUT, map_location=DEVICE, weights_only=True))
model.eval()

show = (val_set or train_set)[:4]
fig, axes = plt.subplots(1, len(show), figsize=(5 * len(show), 5))
if len(show) == 1:
    axes = [axes]
prob = None
for ax, (ip, mp, stem) in zip(axes, show):
    x, y = prepare(ip, mp)
    with torch.no_grad():
        prob = torch.sigmoid(model(torch.from_numpy(x)[None, None].to(DEVICE)))
    pred = (prob[0, 0].cpu().numpy() > 0.5).astype(np.uint8)
    base = cv2.resize(cv2.imread(str(ip)), (INPUT, INPUT))[:, :, ::-1].copy()
    for m, color in ((y.astype(np.uint8), (0, 255, 0)), (pred, (255, 0, 0))):
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(base, cnts, -1, color, 2)
    ax.imshow(base)
    ax.set_title(stem, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

if prob is not None:
    print('확률 분포: ' + format(float(prob.min()), '.3f') + ' ~ '
          + format(float(prob.max()), '.3f'))
    print('원본 체크포인트는 0.47~0.72에 뭉쳐 있었다. 0~1로 벌어졌다면 잘 학습된 것이다.')


## 9. 내려받아서 쓰기

내려받은 `Scanner-Detector-ft.pth`를 로컬 저장소의 **`models/` 폴더**에 넣으면
bookscan의 `검출 모델` 목록에 **파인튜닝**이 새로 뜬다.

입력 크기를 384가 아닌 값으로 바꿔 학습했다면, `bookscan/models/catalog.py`의
`FINETUNED.input_size`도 같은 값으로 고쳐야 한다. 다르면 학습할 때와 추론할 때
입력이 어긋나 성능이 조용히 나빠진다.

**내장 알고리즘과 꼭 견줘 보고** 나은 쪽을 쓰자. 파인튜닝했다고 항상 나은 건 아니다.


In [ ]:
files.download(str(OUT))
